In [2]:
import requests

url = "https://puckpedia.com/team/toronto-maple-leafs"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

In [4]:
import re

matches = re.findall(r'x-data="(.*?)"', html, flags=re.DOTALL)

print(len(matches))

for i, m in enumerate(matches):
    print(f"{i:2}: {m[:150]}")

0


In [5]:
print(html[:500])

<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-S4GIXCunu0cZTKqnoIrou0&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-s


In [12]:
from playwright.async_api import async_playwright
import asyncio

async def test():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        print("SUCCESS")
        await browser.close()

await test()

SUCCESS


In [13]:
from playwright.async_api import async_playwright

async def get_html():
    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36"
        )

        await page.goto(
            "https://puckpedia.com/team/toronto-maple-leafs",
            wait_until="networkidle",
            timeout=60000,
        )

        html = await page.content()

        await browser.close()

        return html

html = await get_html()

print(html[:500])

<!DOCTYPE html><html lang="en" dir="ltr" prefix="og: https://ogp.me/ns#" class="js" style="--subnav-height: 57px; scroll-padding-top: 210px;"><head><link href="https://fonts.googleapis.com/css?family=Archivo:400,500|Arimo:400,500|Bitter:400,500|EB+Garamond:400,500|Lato|Libre+Baskervill|Libre+Franklin:400,500|Lora:400,500|Google+Sans:regular,medium:400,500|Material+Icons|Google+Symbols|Merriweather|Montserrat:400,500|Mukta:400,500|Muli:400,500|Nunito:400,500|Open+Sans:400,500,600|Open+Sans+Conden


In [14]:
import re

matches = re.findall(r'x-data="(.*?)"', html, flags=re.DOTALL)

print(f"{len(matches)} Alpine components\n")

for i, m in enumerate(matches):
    print(f"{i:2}: {m[:120]}")

211 Alpine components

 0: 
 1: draftPickSummary([{&quot;draft_pick_id&quot;:&quot;820271&quot;,&quot;draft_year&quot;:&quot;2027&quot;,&quot;draft_roun
 2: draftPickSummary([{&quot;draft_pick_id&quot;:&quot;820281&quot;,&quot;draft_year&quot;:&quot;2028&quot;,&quot;draft_roun
 3: draftPickSummary([{&quot;draft_pick_id&quot;:&quot;820291&quot;,&quot;draft_year&quot;:&quot;2029&quot;,&quot;draft_roun
 4: tab_panels
 5: draftPickDetails({&quot;draft_pick_id&quot;:&quot;120271&quot;,&quot;draft_year&quot;:&quot;2027&quot;,&quot;draft_round
 6: draftPickDetails({&quot;draft_pick_id&quot;:&quot;820271&quot;,&quot;draft_year&quot;:&quot;2027&quot;,&quot;draft_round
 7: draftPickDetails({&quot;draft_pick_id&quot;:&quot;820272&quot;,&quot;draft_year&quot;:&quot;2027&quot;,&quot;draft_round
 8: draftPickDetails({&quot;draft_pick_id&quot;:&quot;3020272&quot;,&quot;draft_year&quot;:&quot;2027&quot;,&quot;draft_roun
 9: draftPickDetails({&quot;draft_pick_id&quot;:&quot;820273&quot;,&quot;draft_yea

In [15]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

components = soup.select("[x-data]")

print(len(components))

for i, c in enumerate(components):
    print(i, c["x-data"][:120])

211
0 
1 draftPickSummary([{"draft_pick_id":"820271","draft_year":"2027","draft_round":"1","overall_position":"5","team_id":"8","
2 draftPickSummary([{"draft_pick_id":"820281","draft_year":"2028","draft_round":"1","overall_position":"5","team_id":"8","
3 draftPickSummary([{"draft_pick_id":"820291","draft_year":"2029","draft_round":"1","overall_position":"5","team_id":"8","
4 tab_panels
5 draftPickDetails({"draft_pick_id":"120271","draft_year":"2027","draft_round":"1","overall_position":"30","team_id":"1","
6 draftPickDetails({"draft_pick_id":"820271","draft_year":"2027","draft_round":"1","overall_position":"5","team_id":"8","t
7 draftPickDetails({"draft_pick_id":"820272","draft_year":"2027","draft_round":"2","overall_position":"37","team_id":"8","
8 draftPickDetails({"draft_pick_id":"3020272","draft_year":"2027","draft_round":"2","overall_position":"46","team_id":"30"
9 draftPickDetails({"draft_pick_id":"820273","draft_year":"2027","draft_round":"3","overall_position":"69","team_id":"8

In [16]:
import re

keywords = [
    "salary",
    "contract",
    "cap",
    "cap_hit",
    "aav",
    "term",
    "ufa",
    "rfa",
    "expires",
    "expiry",
    "salarycap",
    "caphit",
]

for keyword in keywords:
    print(f"{keyword:12} {len(re.findall(keyword, html, flags=re.IGNORECASE))}")

salary       115
contract     258
cap          1176
cap_hit      23
aav          184
term         118
ufa          266
rfa          65
expires      0
expiry       81
salarycap    2
caphit       45


In [17]:
import re

for m in re.finditer(r"cap_hit", html):
    start = max(0, m.start() - 500)
    end = min(len(html), m.end() + 1500)
    print("=" * 100)
    print(html[start:end])
    print("=" * 100)
    break

ot;162&quot;,
        player_role: &quot;all&quot;,
        filters: {
            bio_pos: [&quot;lw&quot;, &quot;c&quot;, &quot;rw&quot;, &quot;d&quot;],
            contract_yrs_left: [0, 0],
            contract_next: [&quot;0&quot;],
            contract_expiry: [&quot;ufa&quot;, &quot;ufa_no_qo&quot;, &quot;ufa_group6&quot;, &quot;rfa&quot;, &quot;rfa_arb&quot;],
            include_buyouts: [&quot;1&quot;],
            stats_gp: [1, 82]
        },
        teamId: 8,
        sortBy: &quot;cap_hit&quot;,
        sortDirection: &quot;DESC&quot;,
        pageSize: 3,
        lazy: true
    })" data-dashboard-mini-block="1" data-season-select-wired="1" data-player-pic-wired="1" data-mini-sort-delegated="1">
    <h3><a href="/players" class="items-center gap-2 mb-2.5 flex"><img loading="lazy" src="https://cdn.puckpedia.com/s/assets/puckpedia_alt.svg" class="w-[90px]" pinger-seen="true"><span>Player Dashboard</span></a></h3>
    <div class="pp_table-wrapper border border-pp-border roun

In [19]:
import asyncio
from playwright.async_api import async_playwright

async def inspect_requests():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        async def handle_response(response):
            url = response.url.lower()
            if any(x in url for x in [
                "api",
                "player",
                "dashboard",
                "contract",
                "team",
                "graphql"
            ]):
                print(response.status, response.url)

        page.on("response", handle_response)

        await page.goto(
            "https://puckpedia.com/team/toronto-maple-leafs",
            wait_until="networkidle"
        )

        await page.wait_for_timeout(5000)
        await browser.close()

await inspect_requests()

307 https://puckpedia.com/team/toronto-maple-leafs
403 https://puckpedia.com/team/toronto-maple-leafs
200 https://challenges.cloudflare.com/turnstile/v0/b/8eb6d5cd556e/api.js?onload=Wfanq7&render=explicit


TimeoutError: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://puckpedia.com/team/toronto-maple-leafs", waiting until "networkidle"


In [20]:
import re

keywords = [
    "Matthews"
]

for keyword in keywords:
    print(f"{keyword:12} {len(re.findall(keyword, html, flags=re.IGNORECASE))}")

Matthews     13


In [24]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

rows = soup.select("tr.group")

print(len(rows))

54


In [26]:
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

records = []

for row in soup.select("tr.group"):

    player = row.select_one('a[href^="/player/"]')
    if player is None:
        continue

    player_name = player.get_text(strip=True)

    position = None
    for span in row.select("span.font-bold"):
        txt = span.get_text(strip=True)
        if txt in {"C", "LW", "RW", "D", "G"}:
            position = txt
            break

    salary_cells = row.select("td[data-sal]")

    for year, td in enumerate(salary_cells, start=1):
        records.append({
            "player": player_name,
            "position": position,
            "year": year,
            "cap_hit": td.get("data-ch"),
            "aav": td.get("data-aav"),
            "total_salary": td.get("data-sal"),
            "signing_bonus": td.get("data-sb"),
        })

df = pd.DataFrame(records)

df

,player,position,year,cap_hit,aav,total_salary,signing_bonus
0,"Matthews, Auston",C,1,"$13,250,000","$13,250,000","$11,080,000","$10,180,000"
1,"Matthews, Auston",C,2,"$13,250,000","$13,250,000","$10,020,000","$9,120,000"
2,"Nylander, William",None,1,"$11,500,000","$11,500,000","$12,500,000","$11,500,000"
3,"Nylander, William",None,2,"$11,500,000","$11,500,000","$11,500,000","$10,500,000"
4,"Nylander, William",None,3,"$11,500,000","$11,500,000","$11,000,000","$5,000,000"
...,...,...,...,...,...,...,...
107,"Mermis, Dakota",None,1,"$812,500","$812,500","$850,000",$0
108,"Akhtyamov, Artur",None,1,"$900,000","$900,000","$850,000",$0
109,"Akhtyamov, Artur",None,2,"$900,000","$900,000","$900,000",$0
110,"Akhtyamov, Artur",None,3,"$900,000","$900,000","$950,000",$0


In [30]:
import re
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

# ---------------------------------------------------------------------
# Find the roster table that actually contains salary information
# ---------------------------------------------------------------------

table = None

for t in soup.select("table.pp_table-roster"):
    if t.select('td[data-sal]'):
        table = t
        break

if table is None:
    raise ValueError("Could not locate contract table.")

records = []

# ---------------------------------------------------------------------
# Parse players
# ---------------------------------------------------------------------

for row in table.select("tr.group"):

    player_link = row.select_one('a[href^="/player/"]')

    if player_link is None:
        continue

    # Player name
    raw_name = player_link.get_text(strip=True)

    if "," in raw_name:
        last, first = [x.strip() for x in raw_name.split(",", 1)]
        player = f"{first} {last}"
    else:
        player = raw_name

    # ----------------------------------------------------------
    # Position / catches
    # ----------------------------------------------------------

    position = None
    catches = None

    first_td = row.find("td")

    for detail in first_td.select("div.text-xs > div"):

        spans = detail.find_all("span")

        if len(spans) < 2:
            continue

        label = spans[0].get_text(strip=True).lower()
        value = spans[-1].get_text(strip=True).upper()

        if label == "pos":
            position = value

        elif label == "catches":
            position = "G"
            catches = value

    # ----------------------------------------------------------
    # Contract years
    # ----------------------------------------------------------

    salary_cells = row.select('td[data-sal]')

    for year, td in enumerate(salary_cells, start=1):

        html_cell = str(td).lower()

        records.append({

            "player": player,

            "position": position,

            "catches": catches,

            "year": year,

            "cap_hit": td.get("data-ch"),

            "aav": td.get("data-aav"),

            "total_salary": td.get("data-sal"),

            "signing_bonus": td.get("data-sb"),

            "performance_bonus_amount": td.get("data-bonus"),

            "no_movement_clause":
                "no movement clause" in html_cell,

            "no_trade_clause":
                (
                    "no trade clause" in html_cell
                    and "modified no trade clause" not in html_cell
                ),

            "modified_no_trade_clause":
                "modified no trade clause" in html_cell,

            "two_way_contract":
                "two-way contract" in html_cell
                or "two way contract" in html_cell,

            "performance_bonus":
                "performance bonus" in html_cell

        })

# ---------------------------------------------------------------------
# DataFrame
# ---------------------------------------------------------------------

df = pd.DataFrame(records)

money_cols = [
    "cap_hit",
    "aav",
    "total_salary",
    "signing_bonus",
    "performance_bonus_amount",
]

for col in money_cols:

    df[col] = (
        df[col]
        .fillna("0")
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace("", "0")
        .astype(int)
    )

df = df[
    [
        "player",
        "position",
        "catches",
        "year",
        "cap_hit",
        "aav",
        "total_salary",
        "signing_bonus",
        "performance_bonus_amount",
        "no_movement_clause",
        "no_trade_clause",
        "modified_no_trade_clause",
        "two_way_contract",
        "performance_bonus",
    ]
]

df

,player,position,catches,year,cap_hit,aav,total_salary,signing_bonus,performance_bonus_amount,no_movement_clause,no_trade_clause,modified_no_trade_clause,two_way_contract,performance_bonus
0,Auston Matthews,C,None,1,13250000,13250000,11080000,10180000,0,True,False,False,False,False
1,Auston Matthews,C,None,2,13250000,13250000,10020000,9120000,0,True,False,False,False,False
2,William Nylander,"C,RW",None,1,11500000,11500000,12500000,11500000,0,True,False,False,False,False
3,William Nylander,"C,RW",None,2,11500000,11500000,11500000,10500000,0,True,False,False,False,False
4,William Nylander,"C,RW",None,3,11500000,11500000,11000000,5000000,0,True,False,False,False,False
5,William Nylander,"C,RW",None,4,11500000,11500000,11000000,5000000,0,True,False,False,False,False
6,William Nylander,"C,RW",None,5,11500000,11500000,10000000,9000000,0,True,False,False,False,False
7,Matthew Knies,"LW,RW",None,1,7750000,7750000,9000000,2000000,0,False,False,False,False,False
8,Matthew Knies,"LW,RW",None,2,7750000,7750000,9000000,2000000,0,False,False,False,False,False
9,Matthew Knies,"LW,RW",None,3,7750000,7750000,7000000,500000,0,False,False,False,False,False


In [32]:
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

# ---------------------------------------------------------------------
# Find every player contract table.
#
# This excludes:
# - the salary-cap summary table
# - the GearGeek equipment tables
#
# A valid contract table must contain:
# - a player link
# - at least one salary cell with data-sal
# ---------------------------------------------------------------------

contract_tables = []

for table in soup.select("table.pp_table-roster"):
    has_players = bool(table.select_one('a[href^="/player/"]'))
    has_salary_data = bool(table.select_one("td[data-sal]"))

    if has_players and has_salary_data:
        contract_tables.append(table)

if not contract_tables:
    raise ValueError("Could not locate any PuckPedia player contract tables.")

print(f"Found {len(contract_tables)} player contract tables.")

records = []

# ---------------------------------------------------------------------
# Parse every player contract table
# ---------------------------------------------------------------------

for table in contract_tables:

    for row in table.select("tbody > tr"):

        player_link = row.select_one('a[href^="/player/"]')

        if player_link is None:
            continue

        # -------------------------------------------------------------
        # Player name
        # Convert "Matthews, Auston" to "Auston Matthews"
        # -------------------------------------------------------------

        raw_name = player_link.get_text(" ", strip=True)

        if "," in raw_name:
            last_name, first_name = [
                value.strip()
                for value in raw_name.split(",", 1)
            ]
            player = f"{first_name} {last_name}"
        else:
            player = raw_name

        # -------------------------------------------------------------
        # Position and goalie catches
        # -------------------------------------------------------------

        position = None
        catches = None

        first_td = row.find("td")

        if first_td is not None:

            for detail in first_td.select("div.text-xs > div"):

                spans = detail.find_all("span")

                if len(spans) < 2:
                    continue

                label = spans[0].get_text(" ", strip=True).lower()
                value = spans[-1].get_text(" ", strip=True).upper()

                if label == "pos":
                    position = value

                elif label == "catches":
                    position = "G"
                    catches = value

        # -------------------------------------------------------------
        # One row per player per contract year
        # -------------------------------------------------------------

        salary_cells = row.select("td[data-sal]")

        for year, td in enumerate(salary_cells, start=1):

            html_cell = str(td).lower()

            no_movement_clause = (
                "no movement clause" in html_cell
            )

            modified_no_trade_clause = (
                "modified no trade clause" in html_cell
            )

            no_trade_clause = (
                "no trade clause" in html_cell
                and not modified_no_trade_clause
            )

            two_way_contract = (
                "two-way contract" in html_cell
                or "two way contract" in html_cell
            )

            performance_bonus = (
                "performance bonus" in html_cell
            )

            records.append({
                "player": player,
                "position": position,
                "catches": catches,
                "year": year,
                "cap_hit": td.get("data-ch"),
                "aav": td.get("data-aav"),
                "total_salary": td.get("data-sal"),
                "signing_bonus": td.get("data-sb"),
                "performance_bonus_amount": td.get("data-bonus"),
                "no_movement_clause": no_movement_clause,
                "no_trade_clause": no_trade_clause,
                "modified_no_trade_clause": modified_no_trade_clause,
                "two_way_contract": two_way_contract,
                "performance_bonus": performance_bonus,
            })

# ---------------------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------------------

if not records:
    raise ValueError(
        "Contract tables were found, but no player salary records were extracted."
    )

df = pd.DataFrame(records)

# ---------------------------------------------------------------------
# Convert money columns from "$13,250,000" into integer 13250000
# ---------------------------------------------------------------------

money_cols = [
    "cap_hit",
    "aav",
    "total_salary",
    "signing_bonus",
    "performance_bonus_amount",
]

for col in money_cols:
    df[col] = (
        df[col]
        .fillna("0")
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("", "0")
        .astype("int64")
    )

# ---------------------------------------------------------------------
# Final column order
# ---------------------------------------------------------------------

df = df[
    [
        "player",
        "position",
        "catches",
        "year",
        "cap_hit",
        "aav",
        "total_salary",
        "signing_bonus",
        "performance_bonus_amount",
        "no_movement_clause",
        "no_trade_clause",
        "modified_no_trade_clause",
        "two_way_contract",
        "performance_bonus",
    ]
].reset_index(drop=True)

df

Found 6 player contract tables.


,player,position,catches,year,cap_hit,aav,total_salary,signing_bonus,performance_bonus_amount,no_movement_clause,no_trade_clause,modified_no_trade_clause,two_way_contract,performance_bonus
0,Auston Matthews,C,None,1,13250000,13250000,11080000,10180000,0,True,False,False,False,False
1,Auston Matthews,C,None,2,13250000,13250000,10020000,9120000,0,True,False,False,False,False
2,William Nylander,"C,RW",None,1,11500000,11500000,12500000,11500000,0,True,False,False,False,False
3,William Nylander,"C,RW",None,2,11500000,11500000,11500000,10500000,0,True,False,False,False,False
4,William Nylander,"C,RW",None,3,11500000,11500000,11000000,5000000,0,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,Dakota Mermis,LD,None,1,812500,812500,850000,0,0,False,False,False,False,False
108,Artur Akhtyamov,G,L,1,900000,900000,850000,0,0,False,False,False,True,False
109,Artur Akhtyamov,G,L,2,900000,900000,900000,0,0,False,False,False,False,False
110,Artur Akhtyamov,G,L,3,900000,900000,950000,0,0,False,False,False,False,False


In [34]:
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


TEAM_URLS = {
    "Toronto Maple Leafs": (
        "https://puckpedia.com/team/toronto-maple-leafs"
    ),
    "Minnesota Wild": (
        "https://puckpedia.com/team/minnesota-wild"
    ),
    "Washington Capitals": (
        "https://puckpedia.com/team/washington-capitals"
    ),
}


def parse_contract_page(html, team, source_url):
    """
    Parse all player contract tables from one PuckPedia team page.
    """

    soup = BeautifulSoup(html, "html.parser")

    # -----------------------------------------------------------------
    # Find every player contract table.
    #
    # This excludes:
    # - the salary-cap summary table
    # - the GearGeek equipment tables
    #
    # A valid contract table must contain:
    # - a player link
    # - at least one salary cell with data-sal
    # -----------------------------------------------------------------

    contract_tables = []

    for table in soup.select("table.pp_table-roster"):
        has_players = bool(
            table.select_one('a[href^="/player/"]')
        )
        has_salary_data = bool(
            table.select_one("td[data-sal]")
        )

        if has_players and has_salary_data:
            contract_tables.append(table)

    if not contract_tables:
        raise ValueError(
            f"Could not locate any contract tables for {team}."
        )

    records = []

    # -----------------------------------------------------------------
    # Parse every player contract table
    # -----------------------------------------------------------------

    for table in contract_tables:

        for row in table.select("tbody > tr"):

            player_link = row.select_one(
                'a[href^="/player/"]'
            )

            if player_link is None:
                continue

            # ---------------------------------------------------------
            # Player name
            # Convert "Matthews, Auston" to "Auston Matthews"
            # ---------------------------------------------------------

            raw_name = player_link.get_text(
                " ",
                strip=True,
            )

            if "," in raw_name:
                last_name, first_name = [
                    value.strip()
                    for value in raw_name.split(",", 1)
                ]
                player = f"{first_name} {last_name}"
            else:
                player = raw_name

            player_url = (
                "https://puckpedia.com"
                + player_link.get("href", "")
            )

            # ---------------------------------------------------------
            # Position and goalie catches
            # ---------------------------------------------------------

            position = None
            catches = None

            first_td = row.find("td")

            if first_td is not None:

                for detail in first_td.select(
                    "div.text-xs > div"
                ):

                    spans = detail.find_all("span")

                    if len(spans) < 2:
                        continue

                    label = (
                        spans[0]
                        .get_text(" ", strip=True)
                        .lower()
                    )

                    value = (
                        spans[-1]
                        .get_text(" ", strip=True)
                        .upper()
                    )

                    if label == "pos":
                        position = value

                    elif label == "catches":
                        position = "G"
                        catches = value

            # ---------------------------------------------------------
            # One row per player per contract year
            # ---------------------------------------------------------

            salary_cells = row.select("td[data-sal]")

            for year, td in enumerate(
                salary_cells,
                start=1,
            ):

                html_cell = str(td).lower()

                no_movement_clause = (
                    "no movement clause"
                    in html_cell
                )

                modified_no_trade_clause = (
                    "modified no trade clause"
                    in html_cell
                )

                no_trade_clause = (
                    "no trade clause" in html_cell
                    and not modified_no_trade_clause
                )

                two_way_contract = (
                    "two-way contract" in html_cell
                    or "two way contract" in html_cell
                )

                performance_bonus = (
                    "performance bonus" in html_cell
                )

                records.append({
                    "team": team,
                    "player": player,
                    "player_url": player_url,
                    "position": position,
                    "catches": catches,
                    "year": year,
                    "cap_hit": td.get("data-ch"),
                    "aav": td.get("data-aav"),
                    "total_salary": td.get("data-sal"),
                    "signing_bonus": td.get("data-sb"),
                    "performance_bonus_amount":
                        td.get("data-bonus"),
                    "no_movement_clause":
                        no_movement_clause,
                    "no_trade_clause":
                        no_trade_clause,
                    "modified_no_trade_clause":
                        modified_no_trade_clause,
                    "two_way_contract":
                        two_way_contract,
                    "performance_bonus":
                        performance_bonus,
                    "source_url": source_url,
                })

    if not records:
        raise ValueError(
            f"Contract tables were found for {team}, "
            "but no records were extracted."
        )

    df = pd.DataFrame(records)

    # -----------------------------------------------------------------
    # Convert money columns from "$13,250,000" to 13250000
    # -----------------------------------------------------------------

    money_cols = [
        "cap_hit",
        "aav",
        "total_salary",
        "signing_bonus",
        "performance_bonus_amount",
    ]

    for col in money_cols:
        df[col] = (
            df[col]
            .fillna("0")
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace("", "0")
            .astype("int64")
        )

    return df


async def fetch_team_html(page, team, url):
    """
    Load one PuckPedia team page and return its real HTML.
    """

    print(f"Loading {team}...")

    response = await page.goto(
        url,
        wait_until="domcontentloaded",
        timeout=60000,
    )

    if response is not None:
        print(
            f"  HTTP status: {response.status}"
        )

    # Wait for a real salary cell rather than networkidle.
    await page.wait_for_selector(
        "table.pp_table-roster td[data-sal]",
        timeout=60000,
    )

    html = await page.content()

    if "Just a moment..." in html:
        raise RuntimeError(
            f"Cloudflare challenge returned for {team}."
        )

    return html


async def scrape_test_teams():
    """
    Fetch and combine Toronto, Minnesota and Washington.
    """

    team_frames = []
    failures = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 "
                "(Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/139.0.0.0 "
                "Safari/537.36"
            ),
            locale="en-GB",
        )

        page = await context.new_page()

        for team, url in TEAM_URLS.items():

            try:
                html = await fetch_team_html(
                    page=page,
                    team=team,
                    url=url,
                )

                team_df = parse_contract_page(
                    html=html,
                    team=team,
                    source_url=url,
                )

                team_frames.append(team_df)

                print(
                    f"  Extracted "
                    f"{len(team_df):,} contract-year rows "
                    f"for "
                    f"{team_df['player'].nunique():,} players."
                )

            except Exception as exc:
                failures.append({
                    "team": team,
                    "url": url,
                    "error": str(exc),
                })

                print(
                    f"  FAILED: {exc}"
                )

        await context.close()
        await browser.close()

    if not team_frames:
        raise RuntimeError(
            "No team pages were successfully scraped."
        )

    combined_df = pd.concat(
        team_frames,
        ignore_index=True,
    )

    final_columns = [
        "team",
        "player",
        "player_url",
        "position",
        "catches",
        "year",
        "cap_hit",
        "aav",
        "total_salary",
        "signing_bonus",
        "performance_bonus_amount",
        "no_movement_clause",
        "no_trade_clause",
        "modified_no_trade_clause",
        "two_way_contract",
        "performance_bonus",
        "source_url",
    ]

    combined_df = (
        combined_df[final_columns]
        .sort_values(
            ["team", "player", "year"]
        )
        .reset_index(drop=True)
    )

    failures_df = pd.DataFrame(failures)

    return combined_df, failures_df


df, failures_df = await scrape_test_teams()

df

Loading Toronto Maple Leafs...
  HTTP status: 200
  Extracted 112 contract-year rows for 49 players.
Loading Minnesota Wild...
  HTTP status: 200
  Extracted 89 contract-year rows for 47 players.
Loading Washington Capitals...
  HTTP status: 200
  Extracted 114 contract-year rows for 48 players.


,team,player,player_url,position,catches,year,cap_hit,aav,total_salary,signing_bonus,performance_bonus_amount,no_movement_clause,no_trade_clause,modified_no_trade_clause,two_way_contract,performance_bonus,source_url
0,Minnesota Wild,Ben Dexheimer,https://puckpedia.com/player/ben-dexheimer,,None,1,980000,1025000,980000,102500,45000,False,False,False,True,True,https://puckpedia.com/team/minnesota-wild
1,Minnesota Wild,Blake Coleman,https://puckpedia.com/player/blake-coleman,"LW,RW",None,1,2450000,2450000,2450000,0,0,False,False,True,False,False,https://puckpedia.com/team/minnesota-wild
2,Minnesota Wild,Bobby Brink,https://puckpedia.com/player/bobby-brink,RW,None,1,2750000,2750000,2750000,0,0,False,False,False,False,False,https://puckpedia.com/team/minnesota-wild
3,Minnesota Wild,Brock Faber,https://puckpedia.com/player/brock-faber,RD,None,1,8500000,8500000,9500000,0,0,False,False,False,False,False,https://puckpedia.com/team/minnesota-wild
4,Minnesota Wild,Brock Faber,https://puckpedia.com/player/brock-faber,RD,None,2,8500000,8500000,8500000,0,0,False,False,False,False,False,https://puckpedia.com/team/minnesota-wild
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310,Washington Capitals,Tyler Kopff,https://puckpedia.com/player/tyler-kopff,F,None,1,975000,975000,975000,97500,0,False,False,False,True,False,https://puckpedia.com/team/washington-capitals
311,Washington Capitals,Vincent Desharnais,https://puckpedia.com/player/vincent-desharnais,RD,None,1,4200000,4200000,5650000,2000000,0,False,False,False,False,False,https://puckpedia.com/team/washington-capitals
312,Washington Capitals,Vincent Desharnais,https://puckpedia.com/player/vincent-desharnais,RD,None,2,4200000,4200000,4350000,2000000,0,False,False,False,False,False,https://puckpedia.com/team/washington-capitals
313,Washington Capitals,Vincent Desharnais,https://puckpedia.com/player/vincent-desharnais,RD,None,3,4200000,4200000,3400000,2000000,0,False,False,False,False,False,https://puckpedia.com/team/washington-capitals


In [36]:
summary = (
    df.groupby("team")
      .agg(
          players=("player", "nunique"),
          contract_year_rows=("player", "size"),
          unique_goalies=("player", lambda s: s[df.loc[s.index, "position"] == "G"].nunique()),
          missing_positions=("position", lambda s: s.isna().sum()),
          max_contract_year=("year", "max"),
      )
      .reset_index()
)

summary

,team,players,contract_year_rows,unique_goalies,missing_positions,max_contract_year
0,Minnesota Wild,47,89,6,0,5
1,Toronto Maple Leafs,49,112,4,0,5
2,Washington Capitals,48,114,4,0,5


In [40]:
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

BASE_URL = "https://puckpedia.com"
TEAMS_URL = f"{BASE_URL}/teams"


async def get_html(url):

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/138.0.0.0 Safari/537.36"
            )
        )

        await page.goto(url, wait_until="domcontentloaded")

        html = await page.content()

        await browser.close()

        return html


# ---------------------------------------------------------------------
# Download Teams page
# ---------------------------------------------------------------------

html = await get_html(TEAMS_URL)

soup = BeautifulSoup(html, "html.parser")

table = soup.select_one("table.pp_table-teams")

if table is None:
    raise ValueError("Could not locate the PuckPedia teams table.")

# ---------------------------------------------------------------------
# Read column names
# ---------------------------------------------------------------------

headers = []

for th in table.select("thead th"):

    text = th.get_text(" ", strip=True)

    headers.append(text)

print(headers)

records = []

# ---------------------------------------------------------------------
# Parse teams
# ---------------------------------------------------------------------

for row in table.select("tbody > tr"):

    team_link = row.select_one('a[href^="/team/"]')

    if team_link is None:
        continue

    slug = team_link["href"].split("/")[-1]

    values = []

    for td in row.find_all("td"):

        # Most money columns store the exact value here
        if td.has_attr("data-extract_ch") and td["data-extract_ch"]:
            values.append(td["data-extract_ch"])

        else:
            values.append(td.get_text(" ", strip=True))

    record = {
        "team": slug,
        "url": BASE_URL + team_link["href"],
    }

    for header, value in zip(headers, values):

        header = (
            header.lower()
            .replace(" ", "_")
            .replace("/", "_")
            .replace("-", "_")
            .replace("__", "_")
        )

        if header == "":
            continue

        record[header] = value

    records.append(record)

df = pd.DataFrame(records)

print(f"{len(df)} teams found")

df

['Team', '', '', 'Proj Cap Hit', 'Proj Cap Space', 'Current Space', 'Deadline Space', 'Dead Space', 'Active Roster', 'Retained Left', 'Contracts', '2027 Draft Value', 'Average Age', 'Forwards', 'Defense', 'Goalies']
32 teams found


,team,url,proj_cap_hit,proj_cap_space,current_space,deadline_space,dead_space,active_roster,retained_left,contracts,2027_draft_value,average_age,forwards,defense,goalies
0,Vegas Golden Knights,https://puckpedia.com/team/vegas-golden-knights,112661182,-8661182,-8661182,-8661182,0,22 /23,0,49 /50,6.04,30.20,63311182,41100000,8250000
1,Toronto Maple Leafs,https://puckpedia.com/team/toronto-maple-leafs,106752382,-2752382,-2752382,-2752382,0,23 /23,0,49 /50,27.17,30.32,64589280,31413102,10750000
2,Dallas Stars,https://puckpedia.com/team/dallas-stars,105360333,-1360333,-1360333,-1360333,2080000,23 /23,0,45 /50,22.15,29.44,64650000,29330333,9300000
3,Florida Panthers,https://puckpedia.com/team/florida-panthers,103965714,34286,34286,158369,150000,22 /23,0,45 /50,9.15,30.89,68410714,27405000,8000000
4,Washington Capitals,https://puckpedia.com/team/washington-capitals,103924583,75417,75417,348355,0,23 /23,0,47 /50,43.88,28.29,60422500,34652083,8850000
5,Colorado Avalanche,https://puckpedia.com/team/colorado-avalanche,103595841,404159,404159,1866830,2291841,22 /23,0,43 /50,12.74,30.50,62154000,31400000,7750000
6,Minnesota Wild,https://puckpedia.com/team/minnesota-wild,102847499,1152501,1152501,5323457,1666666,23 /23,0,44 /50,16.44,29.87,55605833,35575000,10000000
7,Los Angeles Kings,https://puckpedia.com/team/los-angeles-kings,102225000,1775000,1775000,8198810,600000,23 /23,0,46 /50,22.01,31.45,58250000,35875000,7500000
8,New York Rangers,https://puckpedia.com/team/new-york-rangers,101597024,2402976,2402976,11099461,25000,22 /23,0,45 /50,78.79,27.30,51104167,35900000,14567857
9,Chicago Blackhawks,https://puckpedia.com/team/chicago-blackhawks,101172907,2827093,2827093,13058477,6994999,22 /23,1,43 /50,131.02,26.52,60877908,24716667,8583333
